# Preparation

In this homework, we'll deploy the ride duration model in batch mode. Like in homework 1, we'll use the Yellow Taxi Trip Records dataset.


In [1]:
!pip freeze | grep scikit-learn

scikit-learn==1.7.0


In [2]:
!pip freeze | grep numpy

numpy==2.3.0


In [3]:
!python -V

Python 3.11.9


In [4]:
import pickle
import pandas as pd

In [5]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

/Users/kseniialakhman/projects/mlops/mlops-zoomcamp/.venv_3_11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/kseniialakhman/projects/mlops/mlops-zoomcamp/.venv_3_11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [17]:
# 
url_file = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet'
df = read_data(url_file)

In [18]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

# Q1. Notebook

We'll start with the same notebook we ended up with in homework 1. We cleaned it a little bit and kept only the scoring part. You can find the initial notebook here.

Run this notebook for the March 2023 data.

**What's the standard deviation of the predicted duration for this dataset?**
    
    ✅ 6.24

In [19]:
y_pred.std()

np.float64(6.247488852238704)

# Q2. Preparing the output
Like in the course videos, we want to prepare the dataframe with the output.

First, let's create an artificial ride_id column:
```df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')```

Next, write the ride id and the predictions to a dataframe with results.

Save it as parquet:

```
df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)
```

**What's the size of the output file?**

- 36M
- 46M
- 56M
- 66M

Note: Make sure you use the snippet above for saving the file. It should contain only these two columns. For this question, don't change the dtypes of the columns and use pyarrow, not fastparquet.

     ✅ 66M

In [21]:
def get_ride_id_column(df):
    df['ride_id'] = (
        df['tpep_pickup_datetime'].dt.year.astype(str) + '/' +
        df['tpep_pickup_datetime'].dt.month.astype(str).str.zfill(2) + '_' +
        df.index.astype(str)
    )
    return df['ride_id']

ride_id = get_ride_id_column(df)
ride_id


0                2023/03_0
1                2023/03_1
2                2023/03_2
3                2023/03_3
4                2023/03_4
                ...       
3403761    2023/03_3403761
3403762    2023/03_3403762
3403763    2023/03_3403763
3403764    2023/03_3403764
3403765    2023/03_3403765
Name: ride_id, Length: 3316216, dtype: object

In [22]:
df_result = pd.DataFrame({
    'ride_id': ride_id,
    'y_pred': y_pred
})


In [23]:
output_file = './output/results.parquet'

df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

# Q3. Creating the scoring script

Now let's turn the notebook into a script.

**Which command you need to execute for that?**

    ✅ ```jupyter nbconvert --to script starter.ipynb```

# Q4. Virtual environment

Now let's put everything into a virtual environment. We'll use pipenv for that.

Install all the required libraries. Pay attention to the Scikit-Learn version: it should be the same as in the starter
notebook.

After installing the libraries, pipenv creates two files: `Pipfile`
and `Pipfile.lock`. The `Pipfile.lock` file keeps the hashes of the
dependencies we use for the virtual env.

**What's the first hash for the Scikit-Learn dependency?**

    ✅ sha256:014e07a23fe02e65f9392898143c542a50b6001dbe89cb867e19688e468d049b

# Q5. Parametrize the script

Let's now make the script configurable via CLI. We'll create two 
parameters: year and month.

Run the script for April 2023. 

**What's the mean predicted duration?**

    ✅ 14.29


Hint: just add a print statement to your script.


# Q6. Docker container 

Finally, we'll package the script in the docker container. 
For that, you'll need to use a base image that we prepared. 

This is what the content of this image is:

```dockerfile
FROM python:3.10.13-slim

WORKDIR /app
COPY [ "model2.bin", "model.bin" ]
```

Note: you don't need to run it. We have already done it.

It is pushed to [`agrigorev/zoomcamp-model:mlops-2024-3.10.13-slim`](https://hub.docker.com/layers/agrigorev/zoomcamp-model/mlops-2024-3.10.13-slim/images/sha256-f54535b73a8c3ef91967d5588de57d4e251b22addcbbfb6e71304a91c1c7027f?context=repo),
which you need to use as your base image.

That is, your Dockerfile should start with:

```dockerfile
FROM agrigorev/zoomcamp-model:mlops-2024-3.10.13-slim

# do stuff here
```

This image already has a pickle file with a dictionary vectorizer
and a model. You will need to use them.

Important: don't copy the model to the docker image. You will need
to use the pickle file already in the image. 

Now run the script with docker. 
**What's the mean predicted duration for May 2023?**

    ✅ 0.19


## Bonus: upload the result to the cloud (Not graded)

Just printing the mean duration inside the docker image 
doesn't seem very practical. Typically, after creating the output 
file, we upload it to the cloud storage.

Modify your code to upload the parquet file to S3/GCS/etc.


## Bonus: Use Mage for batch inference

Here we didn't use any orchestration. In practice we usually do.

* Split the code into logical code blocks
* Use Mage to orchestrate the execution



## Publishing the image to dockerhub

This is how we published the image to Docker hub:

```bash
docker build -t mlops-zoomcamp-model:2024-3.10.13-slim .
docker tag mlops-zoomcamp-model:2024-3.10.13-slim agrigorev/zoomcamp-model:mlops-2024-3.10.13-slim

docker login --username USERNAME
docker push agrigorev/zoomcamp-model:mlops-2024-3.10.13-slim
```

This is just for your reference, you don't need to do it.
